# Práctica optativa · EDA aplicado al caso churn

**Recorrido elegido:** SQL + Python  
**Tiempo estimado:** 2 a 3 pomodoros (40 a 60 minutos)  
**Carácter:** optativo; no requiere entrega ni calificación.

> Realizá solamente uno de los dos recorridos equivalentes disponibles: **SQL + Python** o **SQL + R**.

Una empresa de telecomunicaciones quiere comprender por qué algunos clientes abandonan el servicio. En esta práctica vas a transferir los criterios trabajados durante las Semanas 9 y 10 a un dataset diferente: primero reconocerás su estructura, después compararás grupos y finalmente redactarás conclusiones exploratorias prudentes.

El propósito es profundizar y practicar. Esta actividad no reemplaza ni amplía los requisitos de la Entrega 4 del TPI.

## Preparación automática

Ejecutá la celda siguiente sin modificarla. Descargará el dataset público de IBM, preparará las variables y creará la tabla `churn` en DuckDB. Esta preparación técnica no forma parte de los contenidos evaluados.

Cada fila representa un cliente. La variable `Churn` indica si abandonó el servicio (`Yes`) o permaneció (`No`). Algunas variables útiles para explorar son `tenure`, `Contract`, `InternetService`, `TechSupport`, `MonthlyCharges` y `TotalCharges`.

In [ ]:
#@title Preparación automática del entorno (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

for package in ["duckdb", "pandas", "matplotlib"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
DATA_PATH = Path("Telco-Customer-Churn.csv")
if not DATA_PATH.exists():
    urlretrieve(DATA_URL, DATA_PATH)

churn = pd.read_csv(DATA_PATH)
churn["TotalCharges"] = pd.to_numeric(churn["TotalCharges"], errors="coerce")

con = duckdb.connect(":memory:")
con.register("churn_df", churn)
con.execute("CREATE OR REPLACE TABLE churn AS SELECT * FROM churn_df")

print(f"Entorno preparado: {len(churn):,} clientes y {churn.shape[1]} variables.")

## Reconocimiento inicial

Comenzá verificando las dimensiones, la estructura de la tabla y la distribución general de la variable de abandono.

In [ ]:
print(f"Filas: {con.sql('SELECT COUNT(*) FROM churn').fetchone()[0]:,}")
print(f"Columnas: {len(con.sql('DESCRIBE churn').df())}")
con.sql("DESCRIBE churn")

In [ ]:
con.sql("""SELECT
    Churn AS abandono,
    COUNT(*) AS cantidad,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS porcentaje
FROM churn
GROUP BY Churn
ORDER BY Churn""")

## Comparación por tipo de contrato

La tasa de churn expresa qué porcentaje de clientes abandonó dentro de cada tipo de contrato. Esta medida permite comparar grupos de distinto tamaño.

In [ ]:
contratos = con.sql("""SELECT
    Contract AS tipo_contrato,
    COUNT(*) AS clientes,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS abandonos,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        1
    ) AS tasa_churn_pct
FROM churn
GROUP BY Contract
ORDER BY tasa_churn_pct DESC""").df()
contratos

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(contratos["tipo_contrato"], contratos["tasa_churn_pct"], color="#4a7bdc")
plt.xlabel("Tipo de contrato")
plt.ylabel("Tasa de churn (%)")
plt.title("Tasa de churn por tipo de contrato")
plt.xticks(rotation=15)
plt.show()

## Comparación entre quienes abandonaron y quienes permanecieron

Compararemos la antigüedad y los cargos promedio. Interpretá cada diferencia junto con la cantidad de clientes que integra cada grupo.

In [ ]:
con.sql("""SELECT
    Churn AS abandono,
    COUNT(*) AS clientes,
    ROUND(AVG(tenure), 1) AS antiguedad_promedio,
    ROUND(AVG(MonthlyCharges), 2) AS cargo_mensual_promedio,
    ROUND(AVG(TotalCharges), 2) AS cargo_total_promedio
FROM churn
GROUP BY Churn
ORDER BY Churn""")

## Exploración elegida

Elegí **una** de estas preguntas y escribí una consulta SQL para responderla:

1. ¿La tasa de churn cambia según `InternetService`?
2. ¿La tasa de churn cambia entre clientes con y sin `TechSupport`?
3. ¿La antigüedad promedio cambia según `PaymentMethod`?

La consulta debería incluir cantidades o promedios que permitan comparar los grupos. Después podés construir una visualización con Python o R.

In [ ]:
# Escribí acá la consulta SQL correspondiente a la pregunta elegida
resultado_elegido = con.sql("""

""").df()
resultado_elegido

In [ ]:
# Construí acá una visualización del resultado elegido

## Conclusiones

Redactá una conclusión exploratoria breve que responda:

1. ¿Qué diferencias observaste entre quienes abandonaron y quienes permanecieron?
2. ¿Qué patrón te parece más relevante y con qué resultado lo respaldás?
3. ¿Qué acción podría investigar la empresa a partir de ese patrón?
4. ¿Qué no puede concluirse todavía con estos datos?

**Tu conclusión:** escribí acá entre cuatro y seis oraciones.

## Cierre

Esta práctica permite transferir el procedimiento de exploración a un nuevo contexto. Recordá que una asociación observada no demuestra causalidad y que una recomendación empresarial necesita considerar costos, factibilidad e información adicional.

**Fuente del dataset:** IBM, *Telco Customer Churn*, repositorio público `telco-customer-churn-on-icp4d`.  
**Bibliografía:** Downey, A. B. (2025). *Think Stats* (3.ª ed.), capítulo 1; VanderPlas, J. (2016). *Python Data Science Handbook*, capítulos sobre agrupamientos y visualización.

In [ ]:
con.close()